# Iris-3B — LoRA pixel art avec annotations Qwen3.8 Gyro-S

Ce notebook sélectionne les données annotées Qwen3.8 Gyro-S et transporte désormais les palettes exactes mesurées depuis les PNG (hex, alpha, nombre de pixels) dans le manifeste du dataset d’entraînement. Ces données descriptives ne contraignent pas à elles seules les pixels générés; une contrainte de palette doit être appliquée par le décodeur ou le sampler.

Le filtre de départ garde les sprites uniques évalués clean, en pixel art, avec un sujet animé identifiable et un fond transparent ou uni.


## 1. Dépendances et GPU

Dans Colab, sélectionne un runtime GPU avant d’exécuter le notebook. Un runtime avec A100 ou L40S 24 Go ou plus est recommandé. En local, Linux avec CUDA est le chemin le plus simple pour AI Toolkit.

In [1]:
%pip -q install "huggingface_hub>=0.35" pillow matplotlib pyyaml python-dotenv

import json, os, random, re, shutil, subprocess, sys, textwrap
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from dotenv import load_dotenv

for env_file in (Path.cwd() / ".env", Path.cwd().parent / ".env", Path.cwd() / "notebook" / ".env", Path.home() / "iris-sprite-lora" / "training" / ".env"):
    if env_file.is_file():
        load_dotenv(dotenv_path=env_file)
        break

import torch
from PIL import Image, ImageDraw
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU CUDA détecté. Active un runtime GPU dans Colab ou utilise une machine Linux avec NVIDIA CUDA.")

gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16 = torch.cuda.is_bf16_supported()
DTYPE = "bf16" if BF16 else "fp16"
print(f"GPU : {gpu_name} — {gpu_gb:.1f} Go — dtype {DTYPE}")
if gpu_gb < 24:
    print("Attention : moins de 24 Go de VRAM. Le chargement ou l’entraînement peut manquer de mémoire.")

TRAINING_DIR = next((p for p in (Path.cwd(), Path.cwd() / "training", Path.cwd() / "notebook" / "training") if (p / "sprite_taxonomy.py").is_file()), Path.cwd())
if str(TRAINING_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(TRAINING_DIR.resolve()))
WORK_DIR = Path("/content/iris_sprite_lora") if Path("/content").exists() else TRAINING_DIR / "iris_sprite_lora_work"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Dossier de travail :", WORK_DIR.resolve())

Note: you may need to restart the kernel to use updated packages.
GPU : NVIDIA GeForce RTX 5060 Laptop GPU — 7.5 Go — dtype bf16
Attention : moins de 24 Go de VRAM. Le chargement ou l’entraînement peut manquer de mémoire.
Dossier de travail : /home/eth/Documents/Github/iris-pixelart/notebook/training/iris_sprite_lora_work


## 2. Source des annotations Qwen3.8

Par défaut, le notebook utilise le JSONL local s’il existe. Sinon, il cherche sur Hugging Face le snapshot le plus récent dont les lignes contiennent le schéma couleur Qwen3.8 Gyro-S. Tu peux fixer un chemin HF exact dans ANNOTATION_FILE_OVERRIDE.

In [2]:
from huggingface_hub import HfApi, hf_hub_download
import re

DATASET_REPO = "ethanzxv/pixelart-dataset"
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF_ACCESS_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN est absent. Définis-le dans l’environnement ou dans un fichier .env local.")

ANNOTATION_FILE_OVERRIDE = None  # Exemple: annotations/dataset_lab_sync/<run_id>/annotations_v3_4_color_partial.jsonl
QWEN38_LOCAL = next(
    (p / relative
     for p in (TRAINING_DIR, *TRAINING_DIR.parents)
     for relative in (
         "notebook/dataset/taxonomy/taxonomy_work/annotations_v3_4_exact_palette.jsonl",
         "notebook/dataset/taxonomy/taxonomy_work/annotations_v3_4_full_color.jsonl",
     )
     if (p / relative).is_file()),
    None,
)

def is_qwen38_gyro_file(path):
    try:
        with open(path, encoding="utf-8") as stream:
            for line in stream:
                if not line.strip():
                    continue
                row = json.loads(line)
                labels = row.get("labels", {})
                model = str(row.get("model", "")).lower()
                return (
                    "qwen3.8" in model
                    and "gyro" in model
                    and "dominant_color_families" in labels
                    and "visual_quality" in labels
                )
    except (OSError, json.JSONDecodeError):
        return False
    return False

if ANNOTATION_FILE_OVERRIDE:
    annotation_path = hf_hub_download(
        repo_id=DATASET_REPO, repo_type="dataset",
        filename=ANNOTATION_FILE_OVERRIDE, token=HF_TOKEN,
    )
    annotation_source = ANNOTATION_FILE_OVERRIDE
elif QWEN38_LOCAL:
    annotation_path = str(QWEN38_LOCAL)
    annotation_source = "workspace local · " + str(QWEN38_LOCAL)
else:
    api = HfApi(token=HF_TOKEN)
    repo_files = api.list_repo_files(DATASET_REPO, repo_type="dataset")
    candidates = [
        name for name in repo_files
        if (name.startswith("annotations/dataset_lab_sync/") and name.endswith("/annotations_v3_4_color_partial.jsonl"))
        or (name.startswith("annotations/taxonomy_runs/") and name.endswith("/annotations.jsonl"))
    ]
    candidates.sort(
        key=lambda name: name.split("/")[2] if len(name.split("/")) > 2 else "",
        reverse=True,
    )
    annotation_path = None
    annotation_source = None
    for remote_path in candidates:
        local_path = hf_hub_download(
            repo_id=DATASET_REPO, repo_type="dataset",
            filename=remote_path, token=HF_TOKEN,
        )
        if is_qwen38_gyro_file(local_path):
            annotation_path = local_path
            annotation_source = remote_path
            break
    if annotation_path is None:
        raise FileNotFoundError("Aucun snapshot d’annotations Qwen3.8 Gyro-S trouvé sur Hugging Face.")

def read_jsonl_snapshot(path):
    rows = []
    size = Path(path).stat().st_size
    with open(path, "rb") as stream:
        while stream.tell() < size:
            raw = stream.readline(size - stream.tell())
            if not raw.endswith(b"\n"):
                break
            rows.append(json.loads(raw))
    return rows

records = read_jsonl_snapshot(annotation_path)
if not records or not is_qwen38_gyro_file(annotation_path):
    raise ValueError("Le fichier chargé ne contient pas des annotations Qwen3.8 Gyro-S avec taxonomie couleur.")
print(f"{len(records)} annotations Qwen3.8 Gyro-S chargées depuis {annotation_source}")

/home/eth/miniforge3/envs/iris-lora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4918 annotations Qwen3.8 Gyro-S chargées depuis workspace local · /home/eth/Documents/Github/iris-pixelart/notebook/dataset/taxonomy/taxonomy_work/annotations_v3_4_exact_palette.jsonl


## 3. Choisir un sous-ensemble de bonne qualité

Le réglage initial cible les sprites pixel art isolés, propres et lisibles. Tu peux élargir le volume en ajoutant minor_issues, ou changer les sujets/couleurs. Les valeurs unknown et needs_rework sont écartées par défaut.

Les codes hex exacts et les fréquences pixel sont mesurés depuis le PNG et recopiés dans le manifeste d’entraînement, avec une palette séparée pour le canevas d’entraînement sur fond blanc. Ils sont des métadonnées de conditionnement; les inclure dans un prompt ne garantit pas à lui seul une génération limitée à ces couleurs. needs_taxonomy_review reste True sur les annotations automatiques. Pour utiliser les suggestions Qwen telles quelles, garde REQUIRE_HUMAN_APPROVAL=False et vérifie la planche de revue à la cellule suivante. Pour un jeu de données approuvé manuellement, passe le réglage à True et renseigne APPROVED_ASSET_IDS.

In [3]:
ALLOWED_ASSET_TYPES = {"single_sprite"}
ALLOWED_SUBJECT_TYPES = {
    "human_like", "anthropomorphic", "humanoid_robotic", "humanoid_unspecified",
    "animal_like", "insect_like", "arachnid_like", "serpentine", "aquatic_like",
    "amorphous", "plant_like", "mechanical_non_humanoid", "non_humanoid_unspecified",
    "animate_unspecified",
}
ALLOWED_VISUAL_QUALITY = {"clean"}  # Ajoute "minor_issues" pour élargir la sélection.
ALLOWED_RENDER_FAMILIES = {"pixel_art"}
ALLOWED_BACKGROUNDS = {"transparent", "solid_color"}
ALLOWED_PALETTE_SIZES = None  # Exemple: {"very_limited", "limited", "moderate"}
REQUIRED_COLOR_FAMILY = None  # Exemple: "blue", "green", "red"; None pour ne pas filtrer par couleur.
REQUIRE_HUMAN_APPROVAL = False
APPROVED_ASSET_IDS = set()  # Utilisé si REQUIRE_HUMAN_APPROVAL=True.
MAX_TRAIN_IMAGES = None
REJECT_ASSET_IDS = set()
SEED = 42

from collections import Counter
from sprite_taxonomy import visual_hierarchy, clean_caption, prefix

def category_hierarchy(row):
    labels = row.get("labels", {})
    return visual_hierarchy(labels.get("view"), labels.get("pose"))

excluded = Counter()
candidates = []
for row in records:
    labels = row.get("labels", {})
    if row.get("source", {}).get("root", "data") != "data":
        excluded["source_sheets"] += 1
        continue
    if labels.get("asset_type") not in ALLOWED_ASSET_TYPES:
        excluded["asset_type"] += 1
        continue
    if labels.get("subject_type") not in ALLOWED_SUBJECT_TYPES:
        excluded["subject_type"] += 1
        continue
    if labels.get("visual_quality") not in ALLOWED_VISUAL_QUALITY or row.get("need_rework") is not False:
        excluded["quality_or_rework"] += 1
        continue
    if labels.get("render_family") not in ALLOWED_RENDER_FAMILIES:
        excluded["render_family"] += 1
        continue
    if labels.get("background_type") not in ALLOWED_BACKGROUNDS:
        excluded["background"] += 1
        continue
    if ALLOWED_PALETTE_SIZES is not None and labels.get("palette_size") not in ALLOWED_PALETTE_SIZES:
        excluded["palette_size"] += 1
        continue
    colors = labels.get("dominant_color_families", [])
    if not isinstance(colors, list) or not colors or any(c in {"unknown", "not_applicable"} for c in colors):
        excluded["color_labels"] += 1
        continue
    if REQUIRED_COLOR_FAMILY and REQUIRED_COLOR_FAMILY not in colors:
        excluded["color_family"] += 1
        continue
    if REQUIRE_HUMAN_APPROVAL and (
        row.get("review_status") != "human_approved"
        or row.get("asset_id") not in APPROVED_ASSET_IDS
    ):
        excluded["not_human_approved"] += 1
        continue
    if not str(row.get("caption_candidate", "")).strip():
        excluded["empty_caption"] += 1
        continue
    candidates.append(row)

candidates.sort(key=lambda r: r.get("asset_id", ""))
random.Random(SEED).shuffle(candidates)
candidates = [r for r in candidates if r.get("asset_id") not in REJECT_ASSET_IDS]
if MAX_TRAIN_IMAGES is not None:
    candidates = candidates[:MAX_TRAIN_IMAGES]

print("Sources chargées :", len(records))
print("Exclusions par règle :", dict(excluded))
print(f"Candidats retenus : {len(candidates)}")
print("Qualité :", dict(Counter(r["labels"]["visual_quality"] for r in candidates)))
print("Familles de couleur :", dict(Counter(c for r in candidates for c in r["labels"]["dominant_color_families"])))
print("Statut des annotations :", sorted({r.get("review_status", "unknown") for r in candidates}))
print("Palettes hex exactes :", sum(r.get("color_measurements", {}).get("status") == "measured" for r in candidates), "/", len(candidates))
print("Statut des captions :", sorted({r.get("caption_status", "unknown") for r in candidates}))
print("Droits déclarés :", sorted({r.get("rights_status", "unknown") for r in candidates}))
if not candidates:
    raise ValueError("Aucun candidat. Vérifie les filtres ou ajoute minor_issues aux qualités autorisées.")

Sources chargées : 4918
Exclusions par règle : {'subject_type': 716, 'asset_type': 858, 'quality_or_rework': 19, 'background': 26}
Candidats retenus : 3299
Qualité : {'clean': 3299}
Familles de couleur : {'green': 1512, 'orange': 1484, 'yellow': 1697, 'blue': 1728, 'brown': 2398, 'white': 1696, 'pink': 898, 'black': 2065, 'cyan': 377, 'purple': 1047, 'dark_gray': 1299, 'gray': 592, 'light_gray': 144, 'red': 822}
Statut des annotations : ['auto_labeled']
Palettes hex exactes : 3299 / 3299
Statut des captions : ['candidate_unreviewed']
Droits déclarés : ['unknown']


## 4. Télécharger les images retenues

Les images restent dans le cache du runtime et seuls les fichiers retenus sont téléchargés. Cela évite de rapatrier les 100 000 images du dépôt pour ce premier LoRA.

In [ ]:
IMAGE_CACHE = WORK_DIR / "source_images"
IMAGE_CACHE.mkdir(parents=True, exist_ok=True)

def fetch_image(row):
    rel = row["source"]["relative_path"].replace("\\", "/")
    filename = f"data/{rel}"
    # La copie complète du viewer est partagée entre les runs et évite les appels réseau.
    local_repo = next(
        (root / "dataset-viewer/datasets/pixelart-dataset/.gitrepo" for root in (TRAINING_DIR, *TRAINING_DIR.parents)
         if (root / "dataset-viewer/datasets/pixelart-dataset/.gitrepo").is_dir()),
        None,
    )
    paths = [IMAGE_CACHE / filename]
    if local_repo:
        paths.insert(0, local_repo / filename)
    for path in paths:
        if path.is_file():
            with path.open("rb") as stream:
                if stream.read(8) == b"\x89PNG\r\n\x1a\n":
                    return row["asset_id"], path
    # Le cache Hugging Face est consulté sans vérifier le Hub pour chaque image.
    try:
        local_path = hf_hub_download(repo_id=DATASET_REPO, repo_type="dataset", filename=filename,
                                     token=HF_TOKEN, local_files_only=True)
    except Exception:
        local_path = hf_hub_download(repo_id=DATASET_REPO, repo_type="dataset", filename=filename,
                                     token=HF_TOKEN, local_dir=IMAGE_CACHE)
    return row["asset_id"], Path(local_path)

image_paths = {}
errors = []
with ThreadPoolExecutor(max_workers=6) as pool:
    futures = [pool.submit(fetch_image, row) for row in candidates]
    for future in as_completed(futures):
        try:
            asset_id, path = future.result()
            image_paths[asset_id] = path
        except Exception as exc:
            errors.append(str(exc))

if errors:
    print(f"Téléchargements en erreur : {len(errors)}")
    print("Premier message :", errors[0][:500])
print(f"Images locales : {len(image_paths)} / {len(candidates)}")
if not image_paths:
    raise RuntimeError("Aucune image récupérée. Vérifie le token et l’accès au dépôt.")
del HF_TOKEN  # Le jeton n’est plus nécessaire après le téléchargement.

Couldn't access the Hub to check for update but local file already exists. Defaulting to existing file. (error: (Request ID: Root=1-6af5bce6-1ce864097c4fbc15540f5043;18c8099e-7b8a-4c2e-9170-89c5960577e4)

429 Too Many Requests: you have reached your 'resolvers' rate limit.
Retry after 12 seconds (0/5000 requests remaining in current 300s window).
Url: https://huggingface.co/datasets/ethanzxv/pixelart-dataset/resolve/main/data/chrono_trigger/chrono_trigger-magus-0128.png.
We had to rate limit you, you hit the quota of 5000 resolvers requests per 5 minutes period. Upgrade to a PRO user or Team/Enterprise organization account (https://hf.co/pricing) to get higher limits. See https://huggingface.co/docs/hub/rate-limits)
Couldn't access the Hub to check for update but local file already exists. Defaulting to existing file. (error: (Request ID: Root=1-6af5bce6-173308b10b2d26e81f8c9a9c;6d934b42-63ce-492f-9126-f66dd4f12822)

429 Too Many Requests: you have reached your 'resolvers' rate limit.


## 5. Répartition et revue visuelle

La cellule suivante montre les catégories, sous-catégories, rôles, sources, angles de vue, orientations et poses. La planche affiche 36 images avec, pour chacune, sa résolution, son nombre total de pixels, ses pixels visibles et ses couleurs distinctes. Les statistiques portent sur tout le corpus téléchargé. Change `REVIEW_COUNT` pour parcourir davantage d’images.


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import textwrap
from IPython.display import Markdown, display
from PIL import Image, ImageDraw

# Les statistiques portent sur tout le corpus disponible, pas seulement la planche.
distribution_rows = [r for r in candidates if r["asset_id"] in image_paths]
if not distribution_rows:
    raise RuntimeError("Aucune image locale à afficher. Relance d'abord la cellule de téléchargement.")

hierarchy_rows = [(row, category_hierarchy(row)) for row in distribution_rows]
level_1_counts = Counter(h["level_1"] for _, h in hierarchy_rows)
level_2_counts = Counter(f"{h['level_1']} → {h['level_2']}" for _, h in hierarchy_rows)
path_counts = Counter(" → ".join(h["path"]) for _, h in hierarchy_rows)
tree_counts = Counter((h["level_1"], h["level_2"], h["level_3"] or "—") for _, h in hierarchy_rows)


def source_name(row):
    relative_path = row.get("source", {}).get("relative_path", "")
    parts = [part for part in str(relative_path).replace("\\", "/").split("/") if part]
    return parts[0] if parts else "Source inconnue"


def annotation_value(row, label_key):
    value = row.get("labels", {}).get(label_key)
    if value is None or not str(value).strip():
        return "Non renseigné"
    return str(value).strip()


def annotation_counts(label_key):
    return Counter(annotation_value(row, label_key) for row in distribution_rows)


source_counts = Counter(source_name(r) for r in distribution_rows)
total_images = len(distribution_rows)
source_top = source_counts.most_common(12)
other_sources = sum(count for _, count in source_counts.most_common()[12:])
if other_sources:
    source_top.append(("Autres sources", other_sources))


def draw_distribution(ax, counts, title):
    items = list(counts.most_common())[::-1]
    if not items:
        ax.set_title(title, loc="left", fontsize=11, fontweight="bold")
        ax.text(0.5, 0.5, "Aucune donnée", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return
    labels = [name for name, _ in items]
    values = [count for _, count in items]
    bars = ax.barh(labels, values, color="#5686c5")
    ax.set_title(title, loc="left", fontsize=11, fontweight="bold")
    ax.set_xlabel("Images")
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", labelsize=8)
    ax.tick_params(axis="x", labelsize=8)
    ax.set_xlim(0, max(values) * 1.5)
    for bar, count in zip(bars, values):
        ax.text(count + max(values) * 0.025, bar.get_y() + bar.get_height() / 2,
                f"{count}  ({count / total_images:.0%})", va="center", fontsize=8)


fig, axes = plt.subplots(1, 4, figsize=(22, 6), gridspec_kw={"width_ratios": [0.9, 1.35, 1.8, 1.8]})
fig.suptitle(
    f"Répartition des {total_images} images · catégories et sources",
    fontsize=14, fontweight="bold", y=1.02,
)
draw_distribution(axes[0], level_1_counts, "Niveau 1 · catégorie")
draw_distribution(axes[1], level_2_counts, "Niveau 2 · sous-catégorie")
draw_distribution(axes[2], path_counts, "Chemin complet · rôle inclus")
draw_distribution(axes[3], Counter(dict(source_top)), "Source · 12 premières + reste")
fig.tight_layout()
display(fig)
plt.close(fig)

view_counts = annotation_counts("view")
facing_counts = annotation_counts("facing_direction")
pose_counts = annotation_counts("pose")
fig, axes = plt.subplots(1, 3, figsize=(19, 6), gridspec_kw={"width_ratios": [1, 1.25, 1.1]})
fig.suptitle("Répartition des vues, orientations et poses", fontsize=14, fontweight="bold", y=1.02)
draw_distribution(axes[0], view_counts, "Angle de vue · front / side / back")
draw_distribution(axes[1], facing_counts, "Orientation · direction du personnage")
draw_distribution(axes[2], pose_counts, "Pose")
fig.tight_layout()
display(fig)
plt.close(fig)

view_labels = sorted(view_counts)
category_view_counts = Counter(
    (hierarchy["level_2"] or "Non renseigné", annotation_value(row, "view"))
    for row, hierarchy in hierarchy_rows
)
view_table = [
    "### Angles de vue par sous-catégorie",
    "",
    "| Sous-catégorie | " + " | ".join(view_labels) + " | Total |",
    "|---|" + "---:|" * (len(view_labels) + 1),
]
for category in sorted({hierarchy["level_2"] or "Non renseigné" for _, hierarchy in hierarchy_rows}):
    values = [category_view_counts[(category, view)] for view in view_labels]
    view_table.append(
        "| " + category + " | " + " | ".join(str(value) for value in values)
        + " | " + str(sum(values)) + " |"
    )
display(Markdown("\n".join(view_table)))

tree_lines = [
    "### Arborescence des catégories",
    "",
    "| Niveau 1 | Niveau 2 | Niveau 3 / rôle | Images | Part |",
    "|---|---|---|---:|---:|",
]
for (level_1, level_2, level_3), count in sorted(tree_counts.items(), key=lambda item: (-item[1], item[0])):
    tree_lines.append(f"| {level_1} | {level_2} | {level_3} | {count} | {count / total_images:.1%} |")
display(Markdown("\n".join(tree_lines)))

REVIEW_COUNT = min(36, total_images)
review_rows = distribution_rows[:REVIEW_COUNT]
cols = 6
thumb = 96
cell_w, cell_h = 190, 200
rows_n = (len(review_rows) + cols - 1) // cols
sheet = Image.new("RGB", (cols * cell_w, rows_n * cell_h), "#f2f2f2")
draw = ImageDraw.Draw(sheet)


def format_count(value):
    return f"{value:,}".replace(",", " ")


for i, row in enumerate(review_rows):
    x, y = (i % cols) * cell_w, (i // cols) * cell_h
    with Image.open(image_paths[row["asset_id"]]) as source_image:
        im = source_image.convert("RGBA")
    image_width, image_height = im.size
    pixels = im.getdata()
    total_pixels = image_width * image_height
    visible_pixels = sum(1 for pixel in pixels if pixel[3] > 0)
    unique_colors = len({pixel[:3] for pixel in pixels if pixel[3] > 0})
    pixel_info = (
        f"{image_width}×{image_height} · {format_count(total_pixels)} pixels · "
        f"{format_count(visible_pixels)} visibles · {unique_colors} couleurs"
    )

    im.thumbnail((thumb, thumb), Image.Resampling.NEAREST)
    tile = Image.new("RGBA", (thumb, thumb), "white")
    tile.alpha_composite(im, ((thumb - im.width) // 2, (thumb - im.height) // 2))
    sheet.paste(tile.convert("RGB"), (x + (cell_w - thumb) // 2, y + 4))
    draw.multiline_text((x + 5, y + 105), textwrap.fill(pixel_info, width=31), fill="#555555", spacing=1)
    draw.text((x + 5, y + 131), row["asset_id"], fill="black")
    caption = row["caption_candidate"].replace("\n", " ")
    draw.multiline_text((x + 5, y + 147), text=textwrap.fill(caption, width=29)[:78], fill="#333333", spacing=1)

display(sheet)
print("Exemple de caption candidate :", review_rows[0]["caption_candidate"])
print("Pour exclure une ligne, copie son asset_id dans REJECT_ASSET_IDS puis relance les cellules concernées.")


La hiérarchie suit le type sprite, la vue et la pose, sans rôle de jeu. Les annotations automatiques restent à vérifier. La cellule crée un nouveau dossier de paires PNG/caption et un JSONL associé, sans modifier le dataset ni les caches des anciens runs. Les canevas rectangulaires sont arrondis au multiple de 16 supérieur ; les pixels source ne sont ni agrandis, ni réduits.

In [ ]:
from datetime import datetime, timezone

BACKGROUND = (255, 255, 255)
IRIS_MULTIPLE = 16  # Iris-3B utilise des patches de 16 × 16 pixels.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DATASET_OUT = WORK_DIR / "dataset_native" / RUN_ID
DATASET_OUT.mkdir(parents=True, exist_ok=False)

def clean_text(value):
    if value is None or value == "":
        return "not_applicable"
    if isinstance(value, list):
        values = [str(v).strip() for v in value if str(v).strip() and str(v).lower() not in {"unknown", "not_applicable"}]
        return values if values else "not_applicable"
    value = str(value).strip()
    return value if value.lower() not in {"unknown", "not_applicable", "none"} else "not_applicable"

def qwen_tags(row, hierarchy=None):
    labels = row.get("labels", {})
    measurements = row.get("color_measurements", {})
    hierarchy = hierarchy or category_hierarchy(row)
    measured_palette = measurements.get("palette_hex", [])
    training_palette = measurements.get("training_canvas_16", {}).get("palette_hex", [])
    return {
        "asset_type": clean_text(labels.get("asset_type")),
        "subject_type": clean_text(labels.get("subject_type")),
        "body_plan": clean_text(labels.get("body_plan")),
        "action": clean_text(labels.get("action")),
        "visual_quality": clean_text(labels.get("visual_quality")),
        "category_level_1": hierarchy["level_1"],
        "category_level_2": hierarchy["level_2"] or "not_applicable",
        "category_level_3": hierarchy["level_3"] or "not_applicable",
        "category_path": hierarchy["path"],
        "view": clean_text(labels.get("view")),
        "facing": clean_text(labels.get("facing_direction")),
        "pose": clean_text(labels.get("pose")),
        "colors": clean_text(labels.get("dominant_color_families", [])),
        "exact_palette_hex": [color["hex"] for color in measured_palette if color.get("hex")],
        "exact_palette_rgba_hex": [color["rgba_hex"] for color in measured_palette if color.get("rgba_hex")],
        "exact_palette_color_count": measurements.get("distinct_rgba_color_count"),
        "training_canvas_palette_hex": [color["hex"] for color in training_palette if color.get("hex")],
        "palette_size": clean_text(labels.get("palette_size")),
        "palette_temperature": clean_text(labels.get("palette_temperature")),
        "palette_structure": clean_text(labels.get("palette_structure")),
        "palette_contrast": clean_text(labels.get("palette_contrast")),
        "hair": "not_applicable",
        "clothing": "not_applicable",
        "accessories": "not_applicable",
        "equipment": "not_applicable",
        "background": "plain white",
        "rendering_style": "pixel art",
    }

def training_prefix(hierarchy, canvas_size):
    return prefix(hierarchy, canvas_size)

def caption_for_white_background(text):
    text = re.sub(r"(?:against|on) a transparent background", "on a plain white background", text, flags=re.I)
    text = re.sub(r"transparent background", "plain white background", text, flags=re.I)
    text = re.sub(r"\s+", " ", text).strip()
    if "background" not in text.lower():
        text = text.rstrip(" .") + ", on a plain white background."
    return text

def prepare_image(source_path, output_path):
    with Image.open(source_path) as original:
        im = original.convert("RGBA")
    width, height = im.size
    canvas_width = ((width + IRIS_MULTIPLE - 1) // IRIS_MULTIPLE) * IRIS_MULTIPLE
    canvas_height = ((height + IRIS_MULTIPLE - 1) // IRIS_MULTIPLE) * IRIS_MULTIPLE
    canvas = Image.new("RGBA", (canvas_width, canvas_height), (*BACKGROUND, 255))
    canvas.alpha_composite(im, ((canvas_width - width) // 2, (canvas_height - height) // 2))
    canvas.convert("RGB").save(output_path, format="PNG", optimize=True)
    return (width, height), (canvas_width, canvas_height)

training_rows = []
for row in candidates:
    asset_id = row["asset_id"]
    if asset_id not in image_paths or asset_id in REJECT_ASSET_IDS:
        continue
    image_name = f"{asset_id}.png"
    out_image = DATASET_OUT / image_name
    source_size, canvas_size = prepare_image(image_paths[asset_id], out_image)
    hierarchy = category_hierarchy(row)
    free_caption = clean_caption(caption_for_white_background(row["caption_candidate"]))
    tags = qwen_tags(row, hierarchy)
    training_caption = f"{training_prefix(hierarchy, canvas_size)}\n\n{free_caption}"
    (DATASET_OUT / f"{asset_id}.txt").write_text(training_caption, encoding="utf-8")
    training_rows.append({
        "asset_id": asset_id,
        "source": row["source"],
        "category_hierarchy": hierarchy,
        "tags": tags,
        "free_caption": free_caption,
        "training_caption": training_caption,
        "image": image_name,
        "source_size": list(source_size),
        "canvas_size": list(canvas_size),
        "caption_status": row.get("caption_status"),
        "review_status": row.get("review_status"),
        "rights_status": row.get("rights_status", "unknown"),
        "annotation_model": row.get("model"),
        "taxonomy_version": row.get("taxonomy_version"),
        "color_taxonomy_version": row.get("color_taxonomy_version"),
        "visual_quality": row.get("labels", {}).get("visual_quality"),
        "need_rework": row.get("need_rework"),
    })

if not training_rows:
    raise ValueError("Aucune paire image-caption préparée.")
NATIVE_MANIFEST = WORK_DIR / f"qwen3vl_sprite_training_native_{RUN_ID}.jsonl"
with NATIVE_MANIFEST.open("w", encoding="utf-8") as f:
    for row in training_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Dataset préparé : {len(training_rows)} paires image-caption dans {DATASET_OUT}")
print("Canevas :", dict(Counter(f"{r['canvas_size'][0]}x{r['canvas_size'][1]}" for r in training_rows)))
print("Manifeste :", NATIVE_MANIFEST)
print("Répartition niveau 1 :", dict(Counter(row["tags"]["category_level_1"] for row in training_rows)))
print("Répartition niveau 2 :", dict(Counter(f"{row['tags']['category_level_1']} → {row['tags']['category_level_2']}" for row in training_rows)))
print(json.dumps(training_rows[0], ensure_ascii=False, indent=2)[:1800])

In [ ]:
seen = set()
for row in training_rows:
    size = tuple(row["canvas_size"])
    if size in seen:
        continue
    seen.add(size)
    with Image.open(DATASET_OUT / row["image"]) as image:
        preview = image.copy()
    factor = max(1, min(384 // preview.width, 384 // preview.height))
    preview = preview.resize((preview.width * factor, preview.height * factor), Image.Resampling.NEAREST)
    print("Source :", tuple(row["source_size"]), "· canevas :", size)
    display(preview)
    if len(seen) >= 8:
        break

## 7. Installer AI Toolkit

L’intégration Iris-3B est fournie dans AI Toolkit. Le dépôt publie des configurations Iris avec `arch: iris3b`, `quantize_te: true`, `timestep_type: shift` et gradient checkpointing. Le clonage ci-dessous suit la branche principale pour récupérer cette intégration.

In [ ]:
AI_TOOLKIT_DIR = WORK_DIR / "ai-toolkit"
if not AI_TOOLKIT_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/ostris/ai-toolkit.git", str(AI_TOOLKIT_DIR)], check=True)
subprocess.run(["git", "-C", str(AI_TOOLKIT_DIR), "submodule", "update", "--init", "--recursive"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(AI_TOOLKIT_DIR / "requirements.txt")], check=True)
print("AI Toolkit prêt :", AI_TOOLKIT_DIR)

## 8. Configuration et lancement du LoRA

Le run utilise rank 16, batch 1 et 2 000 steps. Chaque exécution crée un nom de run distinct ; les anciens poids sont préservés. Les buckets AI Toolkit ont un pas de 16 et une résolution maximale calculée sur les canevas préparés. Les previews restent en 96 × 96 pour faciliter la comparaison. La taille des plus grands canevas détermine l’utilisation mémoire ; inspecte la répartition avant de lancer l’entraînement.

In [ ]:
import yaml

RUN_NAME = f"iris_pixelart_sprite_native_{RUN_ID}"
MAX_PREPARED_SIDE = max(max(row["canvas_size"]) for row in training_rows)
LEGACY_RUN_NAME = "iris_pixelart_sprite_96"
OUTPUT_DIR = WORK_DIR / "output"
CONFIG_PATH = WORK_DIR / f"{RUN_NAME}.yaml"

SAMPLE_PROMPTS = [
    "humanoid, character, playable, sprite, pixel art, 96x96\n\na playable humanoid character standing with a knife",
    "creature, sprite, pixel art, 96x96\n\na small green creature facing front",
    "humanoid, enemy, sprite, pixel art, 96x96\n\nan enemy character holding a sword",
]

config = {
    "job": "extension",
    "config": {
        "name": RUN_NAME,
        "process": [{
            "type": "sd_trainer",
            "training_folder": str(OUTPUT_DIR),
            "device": "cuda:0",
            "network": {"type": "lora", "linear": 16, "linear_alpha": 16},
            "save": {"dtype": "float16", "save_every": 500, "max_step_saves_to_keep": 3, "push_to_hub": False},
            "datasets": [{
                "folder_path": str(DATASET_OUT),
                "caption_ext": "txt",
                "caption_dropout_rate": 0.02,
                "shuffle_tokens": False,
                "cache_latents_to_disk": True,
                "resolution": [MAX_PREPARED_SIDE],
                "bucket_tolerance": IRIS_MULTIPLE,
                "square_crop": False,
                "random_crop": False,
                "scale": 1.0,
            }],
            "train": {
                "batch_size": 1,
                "steps": 2000,
                "gradient_accumulation_steps": 1,
                "train_unet": True,
                "train_text_encoder": False,
                "gradient_checkpointing": True,
                "cache_text_embeddings": True,
                "noise_scheduler": "flowmatch",
                "timestep_type": "shift",
                "optimizer": "adamw8bit",
                "lr": 0.00005,
                "dtype": DTYPE,
            },
            "model": {
                "arch": "iris3b",
                "name_or_path": "speridlabs/iris-3b",
                "quantize": True,
                "qtype": "qfloat8",
                "quantize_te": True,
                "qtype_te": "qfloat8",
                "low_vram": True,
            },
            "sample": {
                "sampler": "flowmatch",
                "sample_every": 250,
                "width": 96,
                "height": 96,
                "prompts": SAMPLE_PROMPTS,
                "neg": "",
                "seed": 42,
                "walk_seed": False,
                "guidance_scale": 3.0,
                "sample_steps": 30,
            },
        }],
    },
}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
SAMPLE_PROMPTS_PATH = OUTPUT_DIR / RUN_NAME / "sample_prompts.json"
SAMPLE_PROMPTS_PATH.parent.mkdir(parents=True, exist_ok=True)
SAMPLE_PROMPTS_PATH.write_text(json.dumps(SAMPLE_PROMPTS, ensure_ascii=False, indent=2), encoding="utf-8")
print(CONFIG_PATH.read_text(encoding="utf-8"))
print(f"Plus grand côté préparé : {MAX_PREPARED_SIDE} px. Ajuste les données si la VRAM est insuffisante.")

In [ ]:
RUN_TRAINING = False  # Passe à True après avoir vérifié les tailles et la VRAM.
if RUN_TRAINING:
    subprocess.run([sys.executable, "run.py", str(CONFIG_PATH)], cwd=AI_TOOLKIT_DIR, check=True)
else:
    print("Config prête. Pour lancer plus tard : python run.py", CONFIG_PATH)

## 9. Récupérer le LoRA et examiner les previews

Les adapters et images d’échantillonnage sont enregistrés dans un nouveau dossier `output/iris_pixelart_sprite_native_<date>`. Le notebook conserve aussi les previews du run précédent. La galerie ci-dessous associe chaque image à son prompt d’origine. Le nouveau LoRA est séparé des anciens poids pour éviter de reprendre l’entraînement sur une taxonomie différente.

In [ ]:
lora_files = sorted(OUTPUT_DIR.rglob("*.safetensors")) if OUTPUT_DIR.exists() else []
print(f"Fichiers safetensors trouvés : {len(lora_files)}")
for path in lora_files:
    print(path, f"({path.stat().st_size / 1024**2:.1f} MiB)")
print("Dossier de sortie :", OUTPUT_DIR.resolve())

## 10. Exemples de prompts et résultats

La galerie affiche les sorties du checkpoint le plus récent disponible. Après un nouvel entraînement, réexécute la cellule pour actualiser les images et les prompts.

In [ ]:
import base64
import html
import re
from IPython.display import HTML, display

SAMPLE_PATTERN = re.compile(r"__(\d+)_(\d+)\.(?:png|jpe?g|webp)$", re.I)
def sample_files_for(run_name):
    sample_dir = OUTPUT_DIR / run_name / "samples"
    parsed = []
    if sample_dir.exists():
        for path in sample_dir.iterdir():
            match = SAMPLE_PATTERN.search(path.name)
            if match:
                parsed.append((int(match.group(1)), int(match.group(2)), path))
    return parsed

sample_run_name = RUN_NAME
sample_items = sample_files_for(sample_run_name)
if not sample_items:
    sample_run_name = LEGACY_RUN_NAME
    sample_items = sample_files_for(sample_run_name)
if not sample_items:
    raise FileNotFoundError("Aucune image d’échantillonnage trouvée sous le dossier output.")

latest_step = max(step for step, _, _ in sample_items)
latest_by_prompt = {}
for step, prompt_index, image_path in sample_items:
    if step == latest_step and (prompt_index not in latest_by_prompt or image_path.stat().st_mtime > latest_by_prompt[prompt_index].stat().st_mtime):
        latest_by_prompt[prompt_index] = image_path

sample_prompts = None
manifest_path = OUTPUT_DIR / sample_run_name / "sample_prompts.json"
latest_sample_mtime = max(path.stat().st_mtime for path in latest_by_prompt.values())
if manifest_path.exists() and manifest_path.stat().st_mtime <= latest_sample_mtime:
    sample_prompts = json.loads(manifest_path.read_text(encoding="utf-8"))
else:
    output_config_path = OUTPUT_DIR / sample_run_name / "config.yaml"
    if output_config_path.exists():
        output_config = yaml.safe_load(output_config_path.read_text(encoding="utf-8"))
        sample_prompts = output_config.get("config", {}).get("process", [{}])[0].get("sample", {}).get("prompts", [])
if not sample_prompts:
    sample_prompts = SAMPLE_PROMPTS

cards = []
for prompt_index, image_path in sorted(latest_by_prompt.items()):
    prompt = sample_prompts[prompt_index] if prompt_index < len(sample_prompts) else "Prompt non conservé dans le manifeste."
    mime = "image/png" if image_path.suffix.lower() == ".png" else "image/jpeg"
    encoded = base64.b64encode(image_path.read_bytes()).decode("ascii")
    cards.append(
        "<article style='flex:1;min-width:240px;padding:12px;border:1px solid #ddd;border-radius:10px;background:#fff'>"
        f"<h4>Exemple {prompt_index + 1} · étape {latest_step:,}</h4>"
        f"<img src='data:{mime};base64,{encoded}' style='width:192px;height:192px;image-rendering:pixelated;object-fit:contain;background:white;border:1px solid #eee'>"
        f"<pre style='white-space:pre-wrap;text-align:left;font-size:12px;line-height:1.4'>{html.escape(prompt)}</pre>"
        "</article>"
    )
gallery_html = (
    "<div style='display:flex;gap:14px;flex-wrap:wrap;align-items:flex-start'>"
    + "".join(cards)
    + "</div>"
    + f"<p>Run affiché : <code>{html.escape(sample_run_name)}</code></p>"
)
display(HTML(gallery_html))

## Notes

- Le pilote contient des captions `candidate_unreviewed`. Vérifie les captions et les exemples générés avant d’augmenter le nombre de steps.
- Les annotations pilotes signalent des droits `unknown`. Vérifie les licences des sources avant de publier ou redistribuer un LoRA entraîné dessus.
- Les exemples gardent leurs dimensions et pixels natifs ; seul un fond blanc est ajouté jusqu’au prochain multiple de 16. Les anciens fichiers et checkpoints restent séparés.
- Iris-3B est un modèle RGB sans alpha. Pour produire de la transparence, il faut détourer les sorties après génération et inspecter les contours.
- Le pilote de 1 000 images est un point de départ. La sélection actuelle contient environ 500 exemples ; pour un modèle plus général, il faudra captionner et vérifier davantage de sprites du dataset complet.
- Dans les captions candidates, aucun exemple ne mentionne un couteau (1 dague, 2 lames, 289 épées au dernier contrôle). Ajoute des exemples de couteaux captionnés si cette caractéristique doit être fiable.